# Pha 2 — Data Preparation & Validation Strategy: Telco Customer Churn

Input: `df = load_clean()` từ `src/data/loader.py` (đã xác nhận sạch ở Pha 1 — không còn missing value,
`customerID` đã bỏ, `SeniorCitizen` đã chuẩn hóa).

Output của notebook này: một holdout test set **persisted trên đĩa** (bất biến từ đây tới Pha 4), một chiến
lược CV cho Pha 3, một preprocessing pipeline tái sử dụng được, và một con số class-imbalance ratio.

**Chưa train model nào ở đây** — đó là phạm vi của `03_modeling.ipynb`.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.churn_classification.data_split import RANDOM_STATE, TEST_SIZE, get_split
from src.churn_classification.preprocessing import (
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    build_preprocessor,
    compute_scale_pos_weight,
    split_X_y,
)


## 1. Train / Holdout split — vì sao tách riêng khỏi CV

Nếu dùng toàn bộ CV score (đã lặp qua nhiều fold) vừa để **chọn model/tune hyperparameter** vừa để **báo
cáo performance cuối cùng**, con số cuối sẽ lạc quan giả tạo (selection bias): ta đã "nhìn" toàn bộ dữ liệu
qua nhiều vòng thử nghiệm trước khi chốt số liệu. Cách chuẩn: cắt riêng một **holdout test set không đụng
tới** cho tới tận Pha 4, mọi việc chọn model/tune ở Pha 3 chỉ được nhìn thấy phần train pool còn lại.

Split được **ghi ra đĩa** (`data/processed/churn_classification/{train,test}.csv`) thay vì gọi lại
`train_test_split` mỗi lần chạy notebook — nếu không, một lần re-run với version sklearn/pandas khác hoặc
thứ tự dòng dữ liệu đổi có thể vô tình đổi holdout, để lộ khách hàng từng bị giữ lại lọt vào tập train ở
một notebook sau. `get_split()` tự tạo lần đầu, các lần sau chỉ load lại y nguyên.


In [2]:
train_df, test_df = get_split()
print(f"train_df: {train_df.shape}  ({TEST_SIZE:.0%} held out -> test_df: {test_df.shape})")
print(f"random_state={RANDOM_STATE}")

print()
print("Ty le Churn - train:")
print(train_df['Churn'].value_counts(normalize=True).round(4))
print("Ty le Churn - test:")
print(test_df['Churn'].value_counts(normalize=True).round(4))


Loading cached split from /home/tthhieu/survival-analysis/data/processed/churn_classification (created earlier). Delete this folder and re-run to regenerate with current TEST_SIZE/RANDOM_STATE.
train_df: (5977, 21)  (15% held out -> test_df: (1055, 21))
random_state=42

Ty le Churn - train:
Churn
No     0.7341
Yes    0.2659
Name: proportion, dtype: float64
Ty le Churn - test:
Churn
No     0.7346
Yes    0.2654
Name: proportion, dtype: float64


In [3]:
# Leak-check dung customerID (dinh danh that), KHONG dung so sanh full-row: bo du lieu
# nhieu cot categorical + it gia tri numeric ung khien nhieu khach hang khac nhau co the
# trung y het profile mot cach ngau nhien -- full-row match se bao dong gia (da gap khi
# thu ban dau). customerID la thu duy nhat dam bao dung "cung 1 khach hang".
customer_overlap = set(train_df["customerID"]) & set(test_df["customerID"])
assert len(customer_overlap) == 0, "Phat hien customerID trung giua train va test!"
print(f"Xac nhan: 0 customerID trung giua train ({len(train_df)}) va test ({len(test_df)}).")

# Thong tin them (khong phai loi): so dong trung profile hoan toan (bo qua customerID)
# giua train/test -- ky vong > 0 vi ly do neu tren, khong anh huong toi validation strategy.
dup_profile = pd.merge(train_df.drop(columns=["customerID"]), test_df.drop(columns=["customerID"]), how="inner")
print(f"(Thong tin) {len(dup_profile)} dong co feature profile trung nhau giua train/test do trung hop ngau nhien, khac customerID -- khong phai leakage.")


Xac nhan: 0 customerID trung giua train (5977) va test (1055).
(Thong tin) 7 dong co feature profile trung nhau giua train/test do trung hop ngau nhien, khac customerID -- khong phai leakage.


## 2. Cross-validation strategy trong train pool

Đơn vị quan sát là 1 khách hàng tại 1 thời điểm duy nhất, không có time dimension, không có khách hàng lặp
lại (đã xác nhận ở Pha 1: `customerID` unique 100%) → không cần time-series split hay group-aware split.
**StratifiedKFold** là lựa chọn đúng: giữ nguyên tỷ lệ 73/27 ở mỗi fold, quan trọng vì với imbalance này,
một fold ngẫu nhiên (không stratify) có thể vô tình lấy quá ít mẫu Churn=Yes, làm PR-AUC của fold đó nhiễu
mạnh chỉ vì cỡ mẫu dương nhỏ.


In [4]:
X_train, y_train = split_X_y(train_df)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("Kiem tra ty le lop duong (Churn=Yes) o tung fold (phai xap xi 0.265 o moi fold):")
for i, (_, val_idx) in enumerate(cv.split(X_train, y_train)):
    print(f"  fold {i}: n={len(val_idx):4d}  pos_rate={y_train.iloc[val_idx].mean():.4f}")


Kiem tra ty le lop duong (Churn=Yes) o tung fold (phai xap xi 0.265 o moi fold):
  fold 0: n=1196  pos_rate=0.2659
  fold 1: n=1196  pos_rate=0.2659
  fold 2: n=1195  pos_rate=0.2653
  fold 3: n=1195  pos_rate=0.2661
  fold 4: n=1195  pos_rate=0.2661


## 3. Feature typing & phát hiện cấu trúc dữ liệu quan trọng

Danh sách numeric/categorical được khai báo **tường minh** trong `src/churn_classification/preprocessing.py`
(không dùng `df.select_dtypes` tự động) — để pipeline không âm thầm đổi hành vi nếu một cột đổi dtype ở lần
load dữ liệu sau.


In [5]:
# customerID van con trong train_df/test_df (giu de truy vet o Pha 4), nhung khong phai
# feature -> loai tuong minh khoi phep so sanh nay, khong dua vao split_X_y() de "tu dong" loai.
assert set(NUMERIC_FEATURES + CATEGORICAL_FEATURES) == set(train_df.columns) - {"Churn", "customerID"}
print(f"{len(NUMERIC_FEATURES)} numeric, {len(CATEGORICAL_FEATURES)} categorical -> khop du 20 feature + Churn + customerID")

for c in CATEGORICAL_FEATURES:
    print(f"{c:20s} n_unique={train_df[c].nunique()}  {sorted(train_df[c].unique().tolist())}")


3 numeric, 16 categorical -> khop du 20 feature + Churn + customerID
gender               n_unique=2  ['Female', 'Male']
SeniorCitizen        n_unique=2  ['No', 'Yes']
Partner              n_unique=2  ['No', 'Yes']
Dependents           n_unique=2  ['No', 'Yes']
PhoneService         n_unique=2  ['No', 'Yes']
MultipleLines        n_unique=3  ['No', 'No phone service', 'Yes']
InternetService      n_unique=3  ['DSL', 'Fiber optic', 'No']
OnlineSecurity       n_unique=3  ['No', 'No internet service', 'Yes']
OnlineBackup         n_unique=3  ['No', 'No internet service', 'Yes']
DeviceProtection     n_unique=3  ['No', 'No internet service', 'Yes']
TechSupport          n_unique=3  ['No', 'No internet service', 'Yes']
StreamingTV          n_unique=3  ['No', 'No internet service', 'Yes']
StreamingMovies      n_unique=3  ['No', 'No internet service', 'Yes']
Contract             n_unique=3  ['Month-to-month', 'One year', 'Two year']
PaperlessBilling     n_unique=2  ['No', 'Yes']
PaymentMethod      

**Phát hiện quan trọng**: 6 cột (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`,
`StreamingTV`, `StreamingMovies`) dùng giá trị `"No internet service"` thay vì `"No"` bất cứ khi nào
`InternetService == "No"`. Nghĩa là 6 cột này **collinear hoàn hảo** với nhau (và với `InternetService`) tại
đúng những dòng đó — sau one-hot encoding, 6 dummy-column `"...No internet service"` sẽ giống hệt nhau về
mặt giá trị.

Quyết định: **không xử lý đặc biệt ở bước này**. Lý do: `LogisticRegression` mặc định của sklearn dùng L2
regularization, xử lý collinearity hoàn hảo một cách ổn định về số học (chỉ chia đều trọng số giữa các cột
giống nhau, không lỗi/không NaN); tree-based model (Pha 3) hoàn toàn không bị ảnh hưởng bởi collinearity.
Cái giá phải trả là 6 hệ số hồi quy đó sẽ khó diễn giải riêng lẻ — ghi chú lại để **không hiểu sai** khi đọc
feature importance/coefficient ở Pha 3, không phải để implement gì thêm bây giờ.


## 4. Preprocessing pipeline (ColumnTransformer)

Chi tiết lựa chọn (median imputer, StandardScaler, OneHotEncoder với `drop='if_binary'`) đã giải thích trong
docstring của `build_preprocessor()`. Ở đây chỉ fit thử trên `X_train` để xác nhận pipeline chạy không lỗi
và xem shape đầu ra — **không fit trên toàn bộ `df`**, đúng nguyên tắc chống leakage đã đặt ra từ Pha 1.


In [6]:
preprocessor = build_preprocessor()
X_train_transformed = preprocessor.fit_transform(X_train)

print(f"X_train: {X_train.shape} -> sau preprocessing: {X_train_transformed.shape}")
print(f"So chieu tang do OneHotEncoder: {X_train_transformed.shape[1] - X_train.shape[1]} cot moi")
assert not np.isnan(X_train_transformed).any(), "Con NaN sau preprocessing!"
print("Xac nhan: khong con NaN sau preprocessing.")


X_train: (5977, 19) -> sau preprocessing: (5977, 40)
So chieu tang do OneHotEncoder: 21 cot moi
Xac nhan: khong con NaN sau preprocessing.


## 5. Class imbalance handling

Ratio neg/pos được tính **chỉ trên `y_train`** (train pool), không phải toàn bộ `df` hay test set — dù chỉ
là một con số tỷ lệ, tính trên test set vẫn là một dạng leakage (thông tin phân phối của test set rò rỉ vào
lựa chọn cấu hình training).


In [7]:
scale_pos_weight = compute_scale_pos_weight(y_train)
print(f"scale_pos_weight (neg/pos) tren train pool: {scale_pos_weight:.4f}")


scale_pos_weight (neg/pos) tren train pool: 2.7615


**Quyết định xử lý imbalance**: dùng `class_weight='balanced'` (LogisticRegression/RandomForest — sklearn tự
tính nội bộ, không cần con số thủ công) hoặc `scale_pos_weight` (XGBoost/LightGBM — cần truyền thủ công, dùng
đúng con số vừa tính ở trên) ở Pha 3, **không dùng SMOTE**. Lý do:

1. SMOTE tạo mẫu tổng hợp bằng nội suy giữa các điểm lân cận trong không gian feature đã one-hot — với dữ
   liệu phần lớn là categorical, nội suy tuyến tính giữa các one-hot vector tạo ra các điểm "không tồn tại
   trong thực tế nghiệp vụ" (ví dụ giá trị trung gian giữa hai one-hot code), rủi ro overfit vào artefact
   của thuật toán resampling hơn là học pattern churn thật.
2. `class_weight`/`scale_pos_weight` chỉ thay đổi **trọng số trong loss function**, không tạo/xóa dữ liệu —
   đơn giản hơn, không cần đưa `imblearn.pipeline.Pipeline` vào để tránh resampling leak qua CV fold.
3. Về bản chất, cost-sensitivity đã được quyết định xử lý ở tầng metric/threshold (F2-score, Pha 1) — dùng
   `class_weight` ở tầng loss là cách nhất quán để mô hình "biết" trước rằng bỏ sót lớp Yes tốn kém hơn,
   thay vì để mô hình học trên phân phối bị bóp méo rồi mới sửa ở threshold.

Nếu sau khi thử ở Pha 3 mà `class_weight` không đủ cải thiện recall trên lớp Yes, sẽ quay lại cân nhắc SMOTE
như phương án thứ hai — không áp dụng cả hai cùng lúc để giữ được khả năng quy kết nguyên nhân khi so sánh
kết quả.


## 6. Tổng kết Pha 2 & bàn giao sang Pha 3

- Holdout test set: `data/processed/churn_classification/test.csv` (15%, stratified, `random_state=42`) —
  **không được đụng tới cho đến Pha 4**.
- Train pool: `data/processed/churn_classification/train.csv`, dùng `StratifiedKFold(n_splits=5)` cho mọi
  model selection/hyperparameter tuning ở Pha 3.
- Preprocessing: `build_preprocessor()` trong `src/churn_classification/preprocessing.py` — fit **bên trong**
  từng fold CV (qua `sklearn.pipeline.Pipeline` bọc chung với model ở Pha 3), không fit riêng một lần trước.
- Imbalance: xử lý qua `class_weight`/`scale_pos_weight` (tính trên train pool, xem giá trị cụ thể ở cell
  bên trên), không dùng SMOTE.
